# **PREPERING BERITA DARI DETIK.COM**

Pada saat ini adalah proses awal untuk mengolah dataset berita berbahasa Indonesia yang diperoleh melalui proses scraping sehingga siap digunakan dalam analisis teks maupun proses klasifikasi dokumen. Dataset terdiri dari dua kelompok berita, yaitu sport dan finance. Setiap data artikel akan melalui beberapa tahapan pengolahan, mulai dari pengecekan kondisi data, pembersihan teks, konversi teks menjadi data numerik, hingga pengurangan dimensi menggunakan metode Principal Component Analysis (PCA).

Secara umum, tahapan pengolahan data yang dilakukan dalam notebook ini meliputi:
    1. memahami struktur serta kondisi awal dataset;
    2. melakukan preprocessing terhadap teks berita;
    3. menghasilkan representasi fitur dalam bentuk numerik menggunakan beberapa pendekatan; dan
    4. mengurangi jumlah dimensi fitur dengan menggunakan PCA.

Setiap tahapan dilakukan untuk mengubah data teks yang pada awalnya masih berupa data tidak terstruktur menjadi bentuk numerik yang lebih terorganisir. Dengan demikian, data tersebut dapat digunakan dan diolah lebih lanjut oleh berbagai algoritma machine learning.

## **Data Understanding**

Data Understanding merupakan tahap awal yang bertujuan untuk mengetahui karakteristik dan kondisi dataset sebelum masuk ke proses pengolahan lebih lanjut. Pada tahap ini dilakukan pemeriksaan terhadap struktur data, nama dan jenis kolom, jumlah data, serta kategori atau kelas yang terdapat di dalam dataset.

Selain itu, tahap ini juga digunakan untuk mengetahui kualitas data dengan memeriksa kemungkinan adanya missing value, data yang tidak sesuai, maupun baris yang mengalami duplikasi. Pemeriksaan tersebut penting dilakukan agar dataset yang digunakan pada tahap preprocessing dan pemodelan memiliki kondisi yang baik serta sesuai dengan kebutuhan analisis.

In [2]:
import pandas as pd
import re

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import PCA

In [5]:
df_sport = pd.read_csv('data1/crawling_detik_sport_100.csv')
df_finance = pd.read_csv('data1/crawling_detik_finance_100.csv')

In [6]:
df_all_ori = pd.concat([df_sport, df_finance], ignore_index=True)

In [7]:
print(df_all_ori.tail())

     id_berita                                       judul_berita  \
195    8654238  Pertamina Waspada Solar Subsidi 'Bocor', Selis...   
196    8654233  TikTok & Tokopedia Buka Suara soal Kabar Bloki...   
197    8654222  Purbaya Ngaku Tak Pusingkan Anggaran BGN, Tahu...   
198    8654139                   IHSG Ditutup Menguat 1% ke 6.686   
199    8654113  Soal Usulan Nunggak Pajak Jadi Landasan Peramp...   

                                   isi_berita_original kategori_berita  \
195  PT Pertamina Patra Niaga mewaspadai adanya pot...         finance   
196  Tokopedia dan TikTok Shop buka suara terkait l...         finance   
197  Menteri Keuangan (Menkeu) Purbaya Yudhi Sadewa...         finance   
198  Indeks Harga Saham Gabungan (IHSG) ditutup men...         finance   
199  Komisi XI DPR RI mendorong perluasan lingkup p...         finance   

                                            url_berita  
195  https://finance.detik.com/energi/d-8654238/per...  
196  https://finance.detik

In [10]:
df_all = df_all_ori.drop('url_berita', axis=1)

In [11]:
print(df_all.head())

   id_berita                                       judul_berita  \
0    8656258  Men's World Tennis Championship: Rifqi dan Raf...   
1    8656218  Kevin Sanjaya Antusias Lihat Semangat Talenta ...   
2    8656170  Atlet Muda Modern Pentathlon Punya Kesempatan ...   
3    8656176  Romy Tahrizi Juara Nasional Shifter 2026 Bersa...   
4    8656181  Asian Games: Ketum PBTI Yakin Taekwondo Bisa B...   

                                 isi_berita_original kategori_berita  
0  Dua petenis Indonesia, Muhammad Rifqi Fitriadi...           sport  
1  Audisi Umum PB Djarum 2026 sedang berlangsung ...           sport  
2  Modern Pentathlon akan dipertandingkan dalam m...           sport  
3  Kejuaraan Nasional Gokart 2026 yang berlangsun...           sport  
4  Asian Games 2026 tak lama lagi akan dimulai. K...           sport  


In [13]:
print("Distribusi Kelas (Label) pada Dataset:")

jumlah_kelas = df_all["kategori_berita"].value_counts(dropna=False)
persentase_kelas = df_all["kategori_berita"].value_counts(normalize=True, dropna=False).mul(100)

distribusi_kelas = pd.DataFrame({
    "jumlah": jumlah_kelas,
    "persentase": persentase_kelas.round(2)
})

print(distribusi_kelas)

Distribusi Kelas (Label) pada Dataset:
                 jumlah  persentase
kategori_berita                    
sport               100        50.0
finance             100        50.0


In [14]:
print("Pengecekan data kosong (missing values) pada setiap kolom:")
print(df_all.isnull().sum())

Pengecekan data kosong (missing values) pada setiap kolom:
id_berita              0
judul_berita           0
isi_berita_original    0
kategori_berita        0
dtype: int64


In [15]:
print("Pengecekan data duplikat (duplicate values) pada setiap kolom:")
print(df_all.duplicated().sum())

Pengecekan data duplikat (duplicate values) pada setiap kolom:
0


In [17]:
print("Mengecek panjang kata pada setiap baris data:")

kolom_teks = "isi_berita_original"
if kolom_teks not in df_all.columns:
    raise KeyError(
        f"Kolom '{kolom_teks}' tidak ditemukan. "
        f"Kolom yang tersedia: {list(df_all.columns)}"
    )

df_all["jumlah_kata"] = (
    df_all[kolom_teks]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

print("Jumlah kata pada setiap baris:")
print(df_all[["id_berita", "kategori_berita", "jumlah_kata"]])

print("\nStatistik jumlah kata:")
print(df_all["jumlah_kata"].describe())

Mengecek panjang kata pada setiap baris data:
Jumlah kata pada setiap baris:
     id_berita kategori_berita  jumlah_kata
0      8656258           sport          229
1      8656218           sport          266
2      8656170           sport          389
3      8656176           sport          637
4      8656181           sport          221
..         ...             ...          ...
195    8654238         finance          237
196    8654233         finance          452
197    8654222         finance          268
198    8654139         finance          141
199    8654113         finance          300

[200 rows x 3 columns]

Statistik jumlah kata:
count     200.000000
mean      327.575000
std       138.785378
min        22.000000
25%       244.000000
50%       306.000000
75%       374.000000
max      1043.000000
Name: jumlah_kata, dtype: float64


## **Data Preprocessing**

Data berita yang diperoleh dari proses scraping masih dalam bentuk teks mentah sehingga belum dapat langsung digunakan untuk proses analisis maupun klasifikasi. Terdapat berbagai variasi penulisan, seperti penggunaan huruf kapital, tanda baca, angka, imbuhan pada kata, serta kata-kata umum yang kurang memberikan informasi penting bagi model. Oleh karena itu, diperlukan tahap preprocessing untuk membersihkan dan menyeragamkan teks sebelum dilakukan ekstraksi fitur.

Tahapan preprocessing yang diterapkan pada notebook ini terdiri dari beberapa proses berikut:

    1. Pembersihan teks
        Teks diseragamkan dengan mengubah seluruh karakter menjadi huruf kecil, menghilangkan angka dan tanda baca, serta memperbaiki spasi yang berlebihan agar teks menjadi lebih rapi.
    2. Stopword Removal
        Kata-kata yang sering muncul tetapi memiliki kontribusi yang relatif kecil dalam membedakan antar kategori dihilangkan. Contohnya adalah kata penghubung atau kata umum dalam bahasa Indonesia.
    3. Stemming
        Kata yang masih memiliki imbuhan diubah ke bentuk kata dasarnya. Proses ini dilakukan menggunakan library Sastrawi yang dirancang untuk melakukan stemming pada teks berbahasa Indonesia.

Untuk memudahkan proses pengamatan dan perbandingan hasil pada setiap tahapan, beberapa versi teks tetap disimpan dalam dataset. Kolom teks berisi teks awal, sedangkan teks_bersih, teks_tanpa_stopword, dan teks_final masing-masing menunjukkan hasil setelah melalui tahapan pembersihan, penghapusan stopword, dan stemming.

In [18]:
# Inisialisasi Stemmer 
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Inisialisasi Bar Progress
tqdm.pandas()

# inisialisasi StopWordRemoverFactory dan StopWordRemover
factory_stopword = StopWordRemoverFactory()
stopword_remover = factory_stopword.create_stop_word_remover()

# Inisialisasi CountVectorizer dalam mode biner
vectorizer_biner = CountVectorizer(binary=True)

# Inisialisasi CountVectorizer (default untuk menghitung frekuensi/jumlah kata)
vectorizer_frekuensi = CountVectorizer()

# Inisialisasi TfidfTransformer
transformer_tfidf = TfidfTransformer()

In [19]:
def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r'\d+', '', teks) # Menghapus angka
    teks = re.sub(r'[^\w\s]', '', teks) # Menghapus tanda baca
    teks = re.sub(r'\s+', ' ', teks).strip() # Menghapus spasi berlebih
    return teks

In [20]:
# Inisialisasi Stemmer (ditaruh di luar fungsi agar tidak dipanggil berulang kali)
def normalisasi_teks(teks):
    return stemmer.stem(str(teks))

In [21]:
def hapus_stopword(teks):
    return stopword_remover.remove(str(teks))

In [24]:
print("1. Melakukan pembersihan teks (hapus angka, tanda baca, lowercase)...")

kolom_teks = "isi_berita_original"
if kolom_teks not in df_all.columns:
    raise KeyError(
        f"Kolom '{kolom_teks}' tidak ditemukan. "
        f"Kolom yang tersedia: {list(df_all.columns)}"
    )

df_all["teks_bersih"] = df_all[kolom_teks].progress_apply(bersihkan_teks)

print(df_all[[kolom_teks, "teks_bersih"]].head())

1. Melakukan pembersihan teks (hapus angka, tanda baca, lowercase)...


100%|██████████| 200/200 [00:00<00:00, 1140.70it/s]

                                 isi_berita_original  \
0  Dua petenis Indonesia, Muhammad Rifqi Fitriadi...   
1  Audisi Umum PB Djarum 2026 sedang berlangsung ...   
2  Modern Pentathlon akan dipertandingkan dalam m...   
3  Kejuaraan Nasional Gokart 2026 yang berlangsun...   
4  Asian Games 2026 tak lama lagi akan dimulai. K...   

                                         teks_bersih  
0  dua petenis indonesia muhammad rifqi fitriadi ...  
1  audisi umum pb djarum sedang berlangsung di ku...  
2  modern pentathlon akan dipertandingkan dalam m...  
3  kejuaraan nasional gokart yang berlangsung dal...  
4  asian games tak lama lagi akan dimulai ketua u...  


In [25]:
print("2. Menghapus stopword (kata-kata umum yang tidak memiliki makna penting)")
df_all['teks_tanpa_stopword'] = df_all['teks_bersih'].progress_apply(hapus_stopword)

2. Menghapus stopword (kata-kata umum yang tidak memiliki makna penting)


100%|██████████| 200/200 [00:00<00:00, 760.10it/s]


In [26]:
print("3. Melakukan normalisasi (Stemming Sastrawi). Mohon tunggu beberapa saat...")
df_all['teks_final'] = df_all['teks_tanpa_stopword'].progress_apply(normalisasi_teks)

3. Melakukan normalisasi (Stemming Sastrawi). Mohon tunggu beberapa saat...


100%|██████████| 200/200 [21:49<00:00,  6.55s/it]


In [29]:
print("\n=== HASIL PRE-PROCESSING ===")
kolom_hasil = [
    "kategori_berita",
    "isi_berita_original",
    "teks_bersih",
    "teks_tanpa_stopword",
    "teks_final"
]

kolom_tidak_tersedia = [
    kolom for kolom in kolom_hasil
    if kolom not in df_all.columns
]
if kolom_tidak_tersedia:
    raise KeyError(
        f"Kolom hasil preprocessing tidak ditemukan: {kolom_tidak_tersedia}"
    )

display(df_all[kolom_hasil].head())


=== HASIL PRE-PROCESSING ===


,kategori_berita,isi_berita_original,teks_bersih,teks_tanpa_stopword,teks_final
0,sport,"Dua petenis Indonesia, Muhammad Rifqi Fitriadi...",dua petenis indonesia muhammad rifqi fitriadi ...,petenis indonesia muhammad rifqi fitriadi rafa...,tenis indonesia muhammad rifqi fitriadi rafale...
1,sport,Audisi Umum PB Djarum 2026 sedang berlangsung ...,audisi umum pb djarum sedang berlangsung di ku...,audisi umum pb djarum sedang berlangsung kudus...,audisi umum pb djarum sedang langsung kudus ke...
2,sport,Modern Pentathlon akan dipertandingkan dalam m...,modern pentathlon akan dipertandingkan dalam m...,modern pentathlon dipertandingkan momen hari u...,modern pentathlon tanding momen hari ulang tah...
3,sport,Kejuaraan Nasional Gokart 2026 yang berlangsun...,kejuaraan nasional gokart yang berlangsung dal...,kejuaraan nasional gokart berlangsung enam put...,juara nasional gokart langsung enam putar hadi...
4,sport,Asian Games 2026 tak lama lagi akan dimulai. K...,asian games tak lama lagi akan dimulai ketua u...,asian games tak lama dimulai ketua umum pengur...,asi games tak lama mulai ketua umum urus besar...


## **Pembuatan Tabel 1: Binary Document-Term Matrix**

Binary Document-Term Matrix digunakan untuk menggambarkan keberadaan suatu kata pada setiap dokumen. Dalam tabel ini, setiap **baris menunjukkan satu artikel berita**, sedangkan setiap **kolom berisi kata unik** yang ditemukan dalam keseluruhan dokumen.

Nilai pada matriks hanya terdiri dari dua kemungkinan, yaitu:

* `1` menunjukkan bahwa kata tersebut ditemukan dalam dokumen setidaknya satu kali;
* `0` menunjukkan bahwa kata tersebut tidak ditemukan dalam dokumen.

Pada representasi ini, banyaknya suatu kata muncul dalam dokumen tidak diperhitungkan. Artinya, apabila sebuah kata muncul satu kali maupun beberapa kali dalam dokumen yang sama, nilainya tetap `1`. Dengan demikian, dokumen yang memiliki kumpulan kata yang sama akan memperoleh representasi yang sama meskipun jumlah kemunculan masing-masing kata berbeda.

Kolom `label` tidak dimasukkan ke dalam matriks fitur karena kolom tersebut berfungsi sebagai **target atau kelas kategori**. Oleh sebab itu, `label` tetap disimpan secara terpisah dari fitur teks yang digunakan dalam proses analisis.


In [30]:
# Transformasi teks_final menjadi matriks biner
matriks_biner = vectorizer_biner.fit_transform(df_all['teks_final'])

# Mendapatkan daftar kata unik untuk nama kolom
fitur_kata = vectorizer_biner.get_feature_names_out()

# Membuat DataFrame (Tabel 1) dari matriks
df_tabel1 = pd.DataFrame(matriks_biner.toarray(), columns=fitur_kata)

In [33]:
# Menambahkan label kategori di kolom paling akhir
df_tabel1["label"] = df_all["kategori_berita"].values

# Tampilkan hasil Tabel 1
print("=== TABEL 1 (Eksistensi Kata: 0 atau 1) ===")
display(df_tabel1.tail())

=== TABEL 1 (Eksistensi Kata: 0 atau 1) ===


,aadi,aam,aan,aau,abadi,abai,abang,abangpalmerah,abdi,abdul,...,zenix,zhang,zigmars,zona,zoom,zulfikar,zulfikri,zulhas,zulkifli,label
195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
196,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
197,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
198,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
199,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance


In [40]:
print("Jumlah baris dan kolom pada Tabel 1:", df_tabel1.shape)

Jumlah baris dan kolom pada Tabel 1: (200, 5255)


## **Pembuatan Tabel 2: Term Frequency Matrix / Bag of Words**

Metode **Bag of Words (BoW)** digunakan untuk menggambarkan sebuah dokumen berdasarkan frekuensi kemunculan setiap kata di dalamnya. Setiap kata yang ditemukan akan menjadi fitur, sedangkan nilai pada fitur tersebut menunjukkan berapa kali kata tersebut muncul pada suatu artikel.

Berbeda dengan **Binary Document-Term Matrix** yang hanya menggunakan nilai `0` dan `1`, Term Frequency memungkinkan suatu fitur memiliki nilai lebih dari satu. Sebagai contoh, apabila sebuah kata muncul sebanyak tiga kali dalam satu artikel, maka nilai fitur untuk kata tersebut adalah `3`.

Pendekatan ini dapat memberikan informasi mengenai seberapa sering suatu kata digunakan dalam sebuah dokumen. Namun, BoW belum memperhitungkan penyebaran kata tersebut pada dokumen lainnya. Kata yang sering muncul pada hampir seluruh dokumen tetap dapat memiliki nilai frekuensi yang tinggi meskipun sebenarnya kurang efektif untuk membedakan kategori berita.

Matriks yang dihasilkan dari representasi Bag of Words juga umumnya memiliki banyak nilai `0`, karena tidak semua kata muncul pada setiap dokumen. Kondisi tersebut menyebabkan matriks memiliki jumlah dimensi yang besar dan bersifat **sparse**, yaitu sebagian besar elemennya bernilai nol.


In [36]:
# Transformasi teks_final menjadi matriks frekuensi
matriks_frekuensi = vectorizer_frekuensi.fit_transform(df_all['teks_final'])

# Membuat DataFrame (Tabel 2)
df_tabel2 = pd.DataFrame(matriks_frekuensi.toarray(), columns=vectorizer_frekuensi.get_feature_names_out())

In [37]:
# Menambahkan label kategori di kolom paling akhir
df_tabel2["label"] = df_all["kategori_berita"].values

# Tampilkan hasil Tabel 2
print("=== TABEL 2 (Frekuensi Jumlah Kata) ===")
display(df_tabel2.tail())

=== TABEL 2 (Frekuensi Jumlah Kata) ===


,aadi,aam,aan,aau,abadi,abai,abang,abangpalmerah,abdi,abdul,...,zenix,zhang,zigmars,zona,zoom,zulfikar,zulfikri,zulhas,zulkifli,label
195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
196,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
197,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
198,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance
199,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,finance


In [41]:
print("Jumlah baris dan kolom pada Tabel 2:", df_tabel2.shape)

Jumlah baris dan kolom pada Tabel 2: (200, 5255)


## **Pembuatan Tabel 3: Term Frequency-Inverse Document Frequency (TF-IDF)**

TF-IDF (Term Frequency-Inverse Document Frequency) merupakan metode pembobotan yang digunakan untuk menentukan tingkat kepentingan suatu kata dalam sebuah dokumen. Metode ini tidak hanya melihat frekuensi kemunculan kata, tetapi juga mempertimbangkan seberapa banyak dokumen yang mengandung kata tersebut.

Perhitungan TF-IDF terdiri dari dua bagian utama, yaitu:

Term Frequency (TF) menunjukkan tingkat kemunculan suatu kata dalam dokumen tertentu. Semakin sering kata muncul, semakin besar nilai TF-nya.
Inverse Document Frequency (IDF) digunakan untuk memberikan bobot yang lebih kecil pada kata yang banyak ditemukan di berbagai dokumen. Sebaliknya, kata yang hanya muncul pada sebagian kecil dokumen akan memiliki nilai IDF yang lebih tinggi.

Secara matematis, nilai TF-IDF dapat dirumuskan sebagai berikut:

$$ TFIDF(t,d) = TF(t,d) \times IDF(t) $$

Nilai IDF dapat dihitung dengan rumus:

$$ IDF(t) = \log\left(\frac{N}{df(t)}\right) $$

Keterangan:

t = kata atau term yang sedang dihitung;
d = dokumen tempat kata tersebut berada;
N = jumlah keseluruhan dokumen;
df(t) = jumlah dokumen yang mengandung kata t.

Dengan pendekatan ini, kata yang muncul hampir di seluruh dokumen akan mendapatkan bobot yang relatif kecil karena dianggap kurang mampu membedakan dokumen. Sementara itu, kata yang hanya ditemukan pada beberapa dokumen akan mendapatkan bobot yang lebih tinggi sehingga lebih berpotensi menjadi fitur yang membedakan antar kategori berita.

In [42]:
# Mengubah matriks frekuensi (dari Tabel 2) menjadi matriks TF-IDF
matriks_tfidf = transformer_tfidf.fit_transform(matriks_frekuensi)

# Membuat DataFrame (Tabel TF-IDF)
df_tfidf = pd.DataFrame(matriks_tfidf.toarray(), columns=vectorizer_frekuensi.get_feature_names_out())

In [43]:
# Menambahkan label kategori di kolom paling akhir
df_tfidf["label"] = df_all["kategori_berita"].values

# Tampilkan hasil Tabel TF-IDF
print("=== TABEL TF-IDF ===")
display(df_tfidf.tail())

=== TABEL TF-IDF ===


,aadi,aam,aan,aau,abadi,abai,abang,abangpalmerah,abdi,abdul,...,zenix,zhang,zigmars,zona,zoom,zulfikar,zulfikri,zulhas,zulkifli,label
195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,finance
196,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,finance
197,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,finance
198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,finance
199,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,finance


In [44]:
dataset_sebelum_pca = df_tfidf.copy()

## **Pembuatan Tabel 4: Principal Component Analysis (PCA)**

Setelah proses pembentukan TF-IDF selesai, jumlah fitur yang dihasilkan dapat menjadi sangat banyak karena setiap kata unik dalam dataset dapat menjadi sebuah kolom. Kondisi tersebut membuat ukuran data menjadi besar dan lebih sulit untuk diproses. Oleh karena itu, digunakan **Principal Component Analysis (PCA)** untuk menyederhanakan jumlah fitur dengan membentuk beberapa komponen baru yang tetap mewakili sebagian besar informasi dari data asli.

Sebelum menerapkan PCA, data **fitur** dipisahkan terlebih dahulu dari **label**. PCA hanya diterapkan pada fitur yang berbentuk numerik, sedangkan kolom `label` tetap disimpan sebagai penanda kategori dari masing-masing artikel. Dengan demikian, setelah proses reduksi selesai, setiap artikel masih dapat dikaitkan dengan kelas aslinya. Perlu diperhatikan bahwa `PC1`, `PC2`, dan komponen lainnya bukan merupakan kata tertentu, melainkan hasil kombinasi matematis dari berbagai fitur kata.

Secara prinsip, PCA menentukan arah baru yang mampu menjelaskan variasi terbesar pada data. **Komponen utama pertama (PC1)** menangkap variasi paling besar, kemudian **PC2** menangkap variasi terbesar berikutnya dengan arah yang tidak berkorelasi dengan PC1. Proses tersebut dilanjutkan untuk komponen-komponen berikutnya.

Pada notebook ini digunakan `n_components=5`, sehingga data setiap artikel yang sebelumnya memiliki banyak fitur akan direpresentasikan hanya menggunakan **lima komponen utama**, yaitu `PC1` hingga `PC5`.

Penggunaan PCA dapat membuat data menjadi lebih ringkas, mengurangi jumlah perhitungan yang diperlukan, serta membantu ketika data akan divisualisasikan atau digunakan pada tahap pemodelan. Namun, reduksi dimensi juga dapat menyebabkan sebagian informasi dari data awal tidak ikut dipertahankan. Oleh karena itu, pemilihan jumlah komponen sebaiknya mempertimbangkan **explained variance**, yaitu seberapa besar proporsi variasi data yang masih dapat dijelaskan oleh komponen-komponen yang dipilih.


In [45]:
# Memisahkan fitur (kata unik) dan label ('sport'/'finance')
fitur = dataset_sebelum_pca.drop('label', axis=1)
label = dataset_sebelum_pca['label']

In [46]:
# 2. Inisialisasi PCA 
# (Saya set n_components=5 sebagai nilai awal)
pca = PCA(n_components=5)

# Melakukan reduksi dimensi
fitur_pca = pca.fit_transform(fitur)

In [47]:
# 3. Variabel penampungan SESUDAH reduksi
kolom_pca = [f'PC{i+1}' for i in range(fitur_pca.shape[1])]
dataset_sesudah_pca = pd.DataFrame(fitur_pca, columns=kolom_pca)
dataset_sesudah_pca['label'] = label.values

In [48]:
# Tampilkan hasil
print(f"Jumlah kolom sebelum PCA : {dataset_sebelum_pca.shape[1]}")
print(f"Jumlah kolom sesudah PCA : {dataset_sesudah_pca.shape[1]}\n")

Jumlah kolom sebelum PCA : 5255
Jumlah kolom sesudah PCA : 6



## **Hasil Reduksi Dimensi Menggunakan PCA**

Pada proses sebelumnya, dataset memiliki **5.255 kolom fitur** yang berasal dari kata-kata unik hasil ekstraksi teks. Setelah PCA diterapkan dengan `n_components=5`, jumlah fitur tersebut berhasil dikurangi menjadi **5 komponen utama**. Jika kolom `label` tetap dihitung, maka jumlah keseluruhan kolom pada tabel hasil menjadi **6 kolom**, yaitu `PC1`, `PC2`, `PC3`, `PC4`, `PC5`, dan `label`.

Penggunaan lima komponen pada tahap ini dapat dianggap sebagai contoh penerapan reduksi dimensi. Angka `5` tidak memiliki dasar matematis khusus yang menyatakan bahwa lima komponen merupakan jumlah yang paling tepat untuk dataset ini. Jumlah komponen seharusnya disesuaikan dengan tujuan analisis dan seberapa banyak informasi dari data asli yang ingin dipertahankan.

Dalam penerapannya, penentuan nilai `n_components` dapat dilakukan berdasarkan beberapa kebutuhan, antara lain:

* **Untuk visualisasi**, dapat digunakan `n_components=2` atau `n_components=3`. Dengan demikian, data dapat direpresentasikan dalam dua atau tiga dimensi sehingga hubungan atau pola antara artikel dari kategori **sport** dan **finance** dapat divisualisasikan menggunakan scatter plot.

* **Untuk mempertahankan informasi**, jumlah komponen dapat ditentukan berdasarkan nilai **explained variance**. Misalnya, dengan menggunakan `n_components=0.85`, PCA akan memilih jumlah komponen yang diperlukan agar sekitar **85% variasi data** tetap dapat dipertahankan.

* **Untuk kebutuhan pemodelan**, beberapa jumlah komponen dapat diuji, misalnya 50, 100, atau 200. Selanjutnya, performa model machine learning dapat dibandingkan untuk mengetahui pengaruh jumlah dimensi terhadap hasil pemodelan.

Dengan demikian, hasil **5.255 kolom menjadi 6 kolom** menunjukkan bahwa proses reduksi dimensi telah berjalan, tetapi belum dapat dijadikan bukti bahwa lima komponen merupakan jumlah yang paling optimal. Penentuan jumlah komponen yang sesuai perlu didukung oleh tujuan analisis dan evaluasi **explained variance** maupun performa model.


In [49]:
print("=== DATASET SESUDAH PCA ===")
display(dataset_sesudah_pca.tail())

=== DATASET SESUDAH PCA ===


,PC1,PC2,PC3,PC4,PC5,label
195,0.059899,0.013723,-0.045436,-0.108258,-0.014186,finance
196,0.099556,0.020494,-0.029672,-0.110989,0.017762,finance
197,0.240151,0.068716,0.036365,-0.097396,0.004504,finance
198,0.034914,0.028319,-0.083797,-0.289422,0.516356,finance
199,0.087990,0.029642,-0.017148,-0.122204,-0.023276,finance
